In [52]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [53]:
# Load the dataset
df = pd.read_csv("../data/train_augmented.csv")

# Add EOS token
EOS_TOKEN = "<eos>"
input_texts = df["ciphertext"].astype(str).tolist()
target_texts = [text.strip() + EOS_TOKEN for text in df["plaintext"]]

# Tokenizers
input_tokenizer = Tokenizer(char_level=True, lower=False, filters='')
target_tokenizer = Tokenizer(char_level=True, lower=False, filters='', oov_token='<OOV>')

input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts)

# Convert to sequences
input_seqs = input_tokenizer.texts_to_sequences(input_texts)
target_seqs = target_tokenizer.texts_to_sequences(target_texts)

# Padding
max_encoder_seq_length = max(len(seq) for seq in input_seqs)
max_decoder_seq_length = max(len(seq) for seq in target_seqs)

encoder_input_data = pad_sequences(input_seqs, maxlen=max_encoder_seq_length, padding="post")
decoder_full_seq = pad_sequences(target_seqs, maxlen=max_decoder_seq_length, padding="post")

# Shift targets
decoder_input_data = decoder_full_seq[:, :-1]
decoder_target_data = decoder_full_seq[:, 1:]
decoder_target_data = np.expand_dims(decoder_target_data, -1)  # For sparse_categorical_crossentropy

# Train/test split
X_train, X_val, y_train_enc, y_val_enc, y_train_dec, y_val_dec = train_test_split(
    encoder_input_data, decoder_input_data, decoder_target_data, test_size=0.2, random_state=42
)

input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1


In [72]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
enc_emb = Embedding(input_vocab_size, latent_dim, mask_zero=True)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
# Step 1: Define the embedding layer and assign it a name
decoder_embedding = Embedding(
    input_dim=target_vocab_size,
    output_dim=latent_dim,
    mask_zero=True,
    name="decoder_embedding"
)
# Step 2: Use it to generate the embedded decoder input
dec_emb = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Seq2Seq Model
seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
seq2seq_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
seq2seq_model.summary()


Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, None, 256) │     16,384 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_21        │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │      7,424 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_4[0][0… │
│                     │ (None, 256),      │            │ not_equal_21[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ lstm_4[0][1],     │
│                     │ 256), (None,      │            │ lstm_4[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 29)  │      7,453 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,081,885 (4.13 MB)

 Trainable params: 1,081,885 (4.13 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
seq2seq_model.fit(
    [X_train, decoder_input_data],
    decoder_target_data,
    validation_split=0.2,
    batch_size=64,
    epochs=100  # 
)


Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 269ms/step - accuracy: 0.9156 - loss: 0.5838 - val_accuracy: 0.9289 - val_loss: 0.2327
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 268ms/step - accuracy: 0.9335 - loss: 0.1988 - val_accuracy: 0.9273 - val_loss: 0.2364
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.9335 - loss: 0.1886 - val_accuracy: 0.9289 - val_loss: 0.2269
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 246ms/step - accuracy: 0.9329 - loss: 0.1856 - val_accuracy: 0.9273 - val_loss: 0.2498
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 311ms/step - accuracy: 0.9327 - loss: 0.1861 - val_accuracy: 0.9273 - val_loss: 0.2371
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 266ms/step - accuracy: 0.9329 - loss: 0.1847 - val_accuracy: 0.9273 - val_loss: 0.2447
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 291ms/step - accuracy: 0.9334 - loss: 0.1841 - val_accuracy: 0.9273 - val_loss: 0.2395
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 281ms/step - accuracy: 0.9323 - loss: 0.1838 - val_accu

In [73]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Build encoder model for inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inputs for inference
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reuse embedding and LSTM layers from training
decoder_inputs_single = Input(shape=(1,))
dec_emb_inf = decoder_embedding(decoder_inputs_single)
 # chance for error : dec_emb is a tensor, not a layer

decoder_outputs, state_h, state_c = decoder_lstm(dec_emb_inf, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

# Final inference decoder model
decoder_model = Model(
    [decoder_inputs_single] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)


In [74]:
# Reverse lookup
reverse_target_index = {v: k for k, v in target_tokenizer.word_index.items()}
reverse_target_index[0] = ''  # Padding

def decode_sequence(pred_seq, eos_token=EOS_TOKEN, threshold=0.99):
    result = []
    for timestep in pred_seq:
        idx = np.argmax(timestep)
        char = idx2char.get(idx, "")
        if char == eos_token:
            break
        result.append(char)
    return "".join(result)


def evaluate_sample(i):
    input_seq = encoder_input_data[i:i+1]
    decoder_input = np.zeros((1, max_decoder_seq_length - 1))
    output_tokens = seq2seq_model.predict([input_seq, decoder_input])[0]
    print("Ciphertext  :", input_texts[i])
    print("Prediction  :", decode_sequence(output_tokens))
    print("Ground Truth:", target_texts[i].replace(EOS_TOKEN, ""))


In [57]:
from jiwer import wer
import editdistance

def character_error_rate(true, pred):
    return editdistance.eval(true, pred) / len(true)

def evaluate_predictions(n=5):
    for i in range(n):
        input_seq = encoder_input_data[i:i+1]
        decoder_input = np.zeros((1, max_decoder_seq_length - 1))
        output_tokens = seq2seq_model.predict([input_seq, decoder_input])[0]
        predicted = decode_sequence(output_tokens)
        ground_truth = target_texts[i].replace(EOS_TOKEN, "")
        cer = character_error_rate(ground_truth, predicted)
        dist = editdistance.eval(ground_truth, predicted)
        print(f"Sample {i}:")
        print(f"  Ciphertext       : {input_texts[i]}")
        print(f"  Prediction       : {predicted}")
        print(f"  Ground Truth     : {ground_truth}")
        print(f"  CER              : {cer:.4f}")
        print(f"  Levenshtein Dist : {dist}")
        print("-" * 50)

evaluate_predictions(5)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step
Sample 0:
  Ciphertext       : UGYYh VSB  IKREEG LUYNHGR MKSQRR 0
  Prediction       : hiiii     eeeeeeesssssss<<<<<<<<<<<<<<<<
  Ground Truth     : this is a secret message number 0
  CER              : 0.9697
  Levenshtein Dist : 32
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Sample 1:
  Ciphertext       : GqVF VF N FRPERG ZRFFNTR AHORE 1
  Prediction       : hiiii     eeeeeeessssssss52<<<<<<<<<<<<<
  Ground Truth     : this is a secret message number 1
  CER              : 0.9697
  Levenshtein Dist : 32
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Sample 2:
  Ciphertext       : LZAK AK SKWUJWL EWaKdYW FMETWJ 2
  Prediction       : hiiii     eeeeeeesssssss<<<<<<<<<<<<<<<<
  Ground Truth     : this is a secret message number 2
  CER              : 0.9697
  Levenshtein Dist : 32
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62m